# CrisisOps GRPO — Qwen3 4B (Google Colab)

Single path: **Runtime → GPU → Run all**.

1. Fast `pip` install (Unsloth + `trl==0.19.1`, no vLLM).
2. Clone [`vedchamp07/crisisops`](https://github.com/vedchamp07/crisisops) (branch `CRISISOPS_BRANCH`, default `main`).
3. **Qwen3-4B-Base** 4-bit + LoRA, chat template, **CrisisOps** dataset + GRPO (**50** steps by default).
4. Unsloth compatibility: `patch_unsloth_grpo_trainer_vlm_attrs` + **per-step prints** via `GRPOVerboseStepsCallback`.
5. Inference with `model.generate` (no vLLM).

**Requires:** this revision on GitHub so `training/grpo_trainer.py` and `training/grpo_verbose_callback.py` exist in the clone.

**Env:** `CRISISOPS_BRANCH`, `CRISISOPS_MAX_STEPS`, `CRISISOPS_NUM_PROMPTS`, `CRISISOPS_LEVEL`, `CRISISOPS_OUTPUT_DIR`, `CRISISOPS_SEED`, `CRISISOPS_GRPO_BATCH` (default 1 for Colab OOM), `CRISISOPS_GRPO_NUM_GEN`, `CRISISOPS_MAX_COMPLETION`.

If the **session dies with no traceback**, that is usually **GPU RAM OOM** — lower `CRISISOPS_MAX_COMPLETION` (e.g. 128) or keep batch at 1.


### Installation (Colab)

Single `pip` cell: **no vLLM / uv / xformers** (faster on free Colab). Versions align with [`requirements_train.txt`](https://github.com/vedchamp07/crisisops/blob/main/requirements_train.txt).


In [ ]:
import os, subprocess, sys

def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# No vLLM / uv / xformers — keeps this cell much faster than the stock Unsloth Colab block.
os.environ.pop("UNSLOTH_VLLM_STANDBY", None)

pip("packaging", "wheel")
pip("torch>=2.1.0")
pip("transformers==4.56.2", "accelerate>=0.27.0", "peft>=0.10.0")
pip("bitsandbytes>=0.43.0", "datasets>=3.2.0")
pip("unsloth")
pip("trl==0.19.1", "--no-deps")
pip("matplotlib")
print("Install finished. If Unsloth/TRL errors on import: Runtime → Restart session, then skip this cell.")


In [ ]:
# CrisisOps repo (public) — clone into /content and set sys.path
import os, sys, subprocess, shutil, time

try:
    _IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    _IN_COLAB = False

REPO = "https://github.com/vedchamp07/crisisops.git"
BRANCH = os.environ.get("CRISISOPS_BRANCH", "main")
ROOT = "/content/crisisops"

if _IN_COLAB:
    os.chdir("/content")
    if os.path.isdir(ROOT):
        subprocess.run(["rm", "-rf", ROOT], check=False)
        shutil.rmtree(ROOT, ignore_errors=True)
        time.sleep(0.15)
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO, ROOT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(r.stderr or r.stdout)
        raise RuntimeError(
            f"git clone failed (exit {r.returncode}). Try a different BRANCH via CRISISOPS_BRANCH."
        )
    os.chdir(ROOT)
else:
    ROOT = os.path.abspath(os.getcwd())
    if not os.path.isdir(os.path.join(ROOT, "env")):
        raise RuntimeError(f"Expected crisisops repo root; cwd={ROOT}")

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ["CRISISOPS_ROOT"] = ROOT
print("OK —", ROOT)


### Unsloth

### Model + chat template

Run through the chat-template cells below, then **CrisisOps dataset + GRPO** (50 steps).


In [ ]:
import os
# Reduces CUDA OOMs from fragmentation (set before first GPU use in this cell).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
lora_rank = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Base",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model.config.use_cache = False
if getattr(model, "generation_config", None) is not None:
    model.generation_config.use_cache = False

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=lora_rank * 2,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from training.grpo_trainer import prepare_unsloth_grpo_colab
prepare_unsloth_grpo_colab()


### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [ ]:
reasoning_start = "<start_working_out>" # Acts as think-open tag
reasoning_end   = "<end_working_out>"   # Acts as think-close tag
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [ ]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

<a name="Data"></a>
### CrisisOps dataset

Prompts from `training.grpo_trainer.build_training_dataset`; GRPO reward from `_make_reward_fn` (counterfactual `CrisisOpsEnv` rollouts).


In [ ]:
import os, sys
import numpy as np
from datasets import Dataset as HFDataset

root = os.environ.get("CRISISOPS_ROOT", "/content/crisisops")
if root not in sys.path:
    sys.path.insert(0, root)
os.chdir(root)

from env.crisis_generator import CrisisGenerator
from training.grpo_trainer import build_training_dataset, _make_reward_fn

CRISISOPS_LEVEL = int(os.environ.get("CRISISOPS_LEVEL", "1"))
# Trial: fewer prompts than a full run; enough variety for 50 GRPO steps
CRISISOPS_NUM_PROMPTS = int(os.environ.get("CRISISOPS_NUM_PROMPTS", "64"))
CRISISOPS_SEED = int(os.environ.get("CRISISOPS_SEED", "42"))

_crisis_gen = CrisisGenerator(curriculum_level=CRISISOPS_LEVEL)
_rows = build_training_dataset(
    _crisis_gen,
    CRISISOPS_LEVEL,
    n_samples=CRISISOPS_NUM_PROMPTS,
    seed=CRISISOPS_SEED,
)
dataset = HFDataset.from_list(_rows)
print("CrisisOps prompts:", len(dataset), "| columns:", dataset.column_names)

_sample_lens = [
    len(tokenizer.apply_chat_template(dataset[i]["prompt"], tokenize=True))
    for i in range(min(32, len(dataset)))
]
maximum_length = min(1024, int(np.percentile(_sample_lens, 90)) + 64)
print("max_prompt_tokens ~", maximum_length)


<a name="Train"></a>
### Train (CrisisOps GRPO)

Default **50** updates. **`logging_steps=1`** so metrics print every step together with `GRPOVerboseStepsCallback`.


In [ ]:
from training.grpo_trainer import prepare_unsloth_grpo_colab
prepare_unsloth_grpo_colab()

import os
import inspect
from trl import GRPOConfig

TRIAL_GRPO_STEPS = int(os.environ.get("CRISISOPS_MAX_STEPS", "50"))
# Colab often kills the kernel ("session crashed") on GPU OOM with no Python traceback.
# Defaults are conservative; raise CRISISOPS_GRPO_BATCH if you have headroom (nvidia-smi).
_GRPO_BATCH = int(os.environ.get("CRISISOPS_GRPO_BATCH", "1"))
_GRPO_NUM_GEN = int(os.environ.get("CRISISOPS_GRPO_NUM_GEN", "2"))
_MAX_COMP = int(os.environ.get("CRISISOPS_MAX_COMPLETION", "192"))

max_prompt_length = int(min(maximum_length + 1, 1024))
max_completion_length = min(_MAX_COMP, max_seq_length - max_prompt_length)

_sig = set(inspect.signature(GRPOConfig.__init__).parameters.keys())
_cfg_kwargs = dict(
    temperature=0.9,
    learning_rate=2e-5,
    logging_steps=1,
    per_device_train_batch_size=_GRPO_BATCH,
    gradient_accumulation_steps=1,
    num_generations=max(2, _GRPO_NUM_GEN),
    max_prompt_length=max_prompt_length,
    save_steps=max(1, TRIAL_GRPO_STEPS),
    max_steps=TRIAL_GRPO_STEPS,
    report_to="none",
    output_dir=os.environ.get("CRISISOPS_OUTPUT_DIR", "/content/crisisops_grpo_trial"),
)
if "max_completion_length" in _sig:
    _cfg_kwargs["max_completion_length"] = max_completion_length
elif "max_new_tokens" in _sig:
    _cfg_kwargs["max_new_tokens"] = max_completion_length

training_args = GRPOConfig(**_cfg_kwargs)

reward_fn = _make_reward_fn(_crisis_gen, CRISISOPS_LEVEL, model, tokenizer)
print(
    "Trial GRPO: max_steps=", TRIAL_GRPO_STEPS,
    "| batch=", _GRPO_BATCH,
    "| num_gen=", max(2, _GRPO_NUM_GEN),
    "| max_completion=", max_completion_length,
    "| out=", training_args.output_dir,
)


Run the next cell to start training (watch for `[GRPO] update …` and `metrics @ step …` lines).


In [ ]:
from training.grpo_trainer import prepare_unsloth_grpo_colab, patch_unsloth_grpo_trainer_vlm_attrs
prepare_unsloth_grpo_colab()
from trl import GRPOTrainer
from training.grpo_verbose_callback import GRPOVerboseStepsCallback
import inspect as _ins

_tsig = _ins.signature(GRPOTrainer.__init__).parameters
_tkw = dict(
    model=model,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=dataset,
)
if "processing_class" in _tsig:
    _tkw["processing_class"] = tokenizer
else:
    _tkw["tokenizer"] = tokenizer

trainer = GRPOTrainer(**_tkw)
patch_unsloth_grpo_trainer_vlm_attrs(trainer)
trainer.add_callback(GRPOVerboseStepsCallback())
trainer.train()


<a name="Inference"></a>
### Inference
Quick **HF generate** smoke test after GRPO (same LoRA weights in memory).

In [ ]:
import torch
text = "What is the sqrt of 101?"
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        use_cache=False,
    )
tokenizer.decode(out[0], skip_special_tokens=True)


And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test:

In [ ]:
import torch
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        use_cache=False,
    )
tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving weights

Unsloth can merge adapters or export GGUF — see [Unsloth deployment docs](https://unsloth.ai/docs/basics/inference-and-deployment). This notebook uses **HF `generate`** for inference (no vLLM).


In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).